# How does the outcome change with dose?

How does the outcome change with dose?

Estimate a response curve, then ask three follow-up questions:

- How steep is the curve at one dose? Use a local derivative.
- What percentage change in outcome follows a percentage change in dose? Use elasticity.
- What is the average slope across the observed data? Use an average derivative.

For each answer, check whether the graph identifies it, whether the data support
it, and what uncertainty is reported.

The simulated structural curve is E[outcome | do(dose = d)] = 12 + 3 ln d - 0.3 d^2. The
script checks the adjusted curve at dose 4.5 (within 0.6) and the local slope at 2.5 (within
0.5) against it; the elasticity, the average derivative and the dose beyond the data are
reported, not checked. Pointwise intervals are not a simultaneous band.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iridae-dev/antecedent/blob/v3.0.0/examples/notebooks/continuous_causal_response.ipynb)

**Repository:** <https://github.com/iridae-dev/antecedent>  
**Documentation:** <https://antecedent.readthedocs.io/>  
**Script:** [`examples/python/continuous_causal_response.py`](https://github.com/iridae-dev/antecedent/blob/v3.0.0/examples/python/continuous_causal_response.py)

## Setup

The next cell installs Antecedent from PyPI when it is not already present (Google Colab).
Use **Runtime → Run all**. If you upgrade after importing it, restart the kernel before
continuing.

In [ ]:
import importlib.util
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError(
        "Antecedent needs Python 3.11 or later. In Colab: Runtime → Change runtime type."
    )
if importlib.util.find_spec("antecedent") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "antecedent>=3.0.0,<3.1",
            "numpy",
            "pandas",
            "matplotlib",
        ]
    )

import antecedent

if tuple(int(part) for part in antecedent.__version__.split(".")[:2]) != (3, 0):
    raise RuntimeError(
        "This notebook requires a different Antecedent minor. "
        "Upgrade with pip install -U 'antecedent>=3.0.0,<3.1' and restart the kernel."
    )

In [ ]:
import numpy as np
from antecedent import (
    AverageDerivative,
    Elasticity,
    PointDerivative,
    ResponseCurve,
    analyze,
)
from antecedent.outcomes import require_success

SEED = 500
N = 1_500

## A confounded nonlinear dose process

Baseline severity affects both the dose selected by clinicians and the outcome. The graph
therefore requires adjustment for `severity`. Dose is positive, which gives the elasticity a
meaningful treatment scale.

In [ ]:
rng = np.random.default_rng(SEED)
severity = rng.normal(size=N)
dose = np.clip(2.5 + 0.65 * severity + rng.normal(scale=0.65, size=N), 0.25, 5.0)
outcome = (
    12.0 + 3.0 * np.log(dose) - 0.30 * dose**2 + 1.25 * severity + rng.normal(scale=0.45, size=N)
)

data = {"severity": severity, "dose": dose, "outcome": outcome}
graph = [
    ("severity", "dose"),
    ("severity", "outcome"),
    ("dose", "outcome"),
]

print(float(dose.min()), float(np.median(dose)), float(dose.max()))

## Ask four questions

Each query takes the treatment name first and the outcome name second.
For derivatives and elasticities, choose a bandwidth in `estimator_config`;
Silverman's rule is not supported for those queries.

The curve includes a dose of `5.25`, beyond the data's maximum of `5.0`.
Watch for the support warning at that point: requesting a dose does not mean
the data can support an estimate there.

In [ ]:
curve_query = ResponseCurve(
    "dose",
    "outcome",
    grid=[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.25],
)
point_query = PointDerivative("dose", "outcome", at=2.5)
elasticity_query = Elasticity("dose", "outcome", at=2.5)
average_query = AverageDerivative("dose", "outcome", weighting="observed")

curve = require_success(analyze(data, graph=graph, query=curve_query))
point = require_success(
    analyze(data, graph=graph, query=point_query, estimator_config={"bandwidth": 0.35})
)
elasticity = require_success(
    analyze(data, graph=graph, query=elasticity_query, estimator_config={"bandwidth": 0.35})
)
average = require_success(analyze(data, graph=graph, query=average_query))

### Reuse the analysis and read its report

Keep `result.study` to run the same analysis again. `ant.estimate(study)` runs
the prepared branch; `ant.rebind(study, data=new_data)` returns a new branch on the
new data without changing the original. Earlier results stay unchanged.

The report explains the answer and its limitations. Check calibration too:
`not_measured` means no coverage record measured this construction; the interval is
reported without a calibration claim.
A passing diagnostic does not fill that gap.

In [ ]:
study = curve.study
report = curve.inspect().to_dict()
print("Answer:", curve.answer)
print("Calibration:", curve.calibration.status, curve.calibration.reason)
print(report)

## Inspect the orthogonal result axes

An identified estimand can have weak or absent empirical support. Likewise, `pointwise`
uncertainty describes coverage at individual grid points; it is not a simultaneous band for
the whole curve. The provenance operation identifies the implemented algorithm, while
assumptions remain inspectable rather than implicit.

In [ ]:
print(curve.inspect().to_dict())

In [ ]:
curve_rows = [
    {"dose": point_values[0], "mean_response": response_values[0]}
    for point_values, response_values in zip(
        curve.response.points, curve.response.values, strict=True
    )
]
print(curve_rows)

In [ ]:
print(
    {
        "local_derivative_at_2_5": point.estimate,
        "elasticity_at_2_5": elasticity.estimate,
        "observed_law_average_derivative": average.estimate,
        "local_support": point.support.status,
        "average_support": average.support.status,
    }
)

In [ ]:
# The simulation above fixes the structural curve: E[outcome | do(dose = d)] =
# 12 + 3 ln d - 0.3 d^2 (severity has mean zero). Confounding by severity biases the
# unadjusted slope at dose 2.5 to about +0.66, far from the structural -0.3, and lifts the
# unadjusted level at dose 4.5 by about 1.9; the adjusted answers must sit near the truth.
def structural(d):
    return 12.0 + 3.0 * np.log(d) - 0.3 * d**2


by_dose = {
    round(point_values[0], 2): response_values[0]
    for point_values, response_values in zip(
        curve.response.points, curve.response.values, strict=True
    )
}
assert abs(by_dose[4.5] - structural(4.5)) < 0.6, by_dose[4.5]
assert abs(point.estimate - (3.0 / 2.5 - 0.6 * 2.5)) < 0.5, point.estimate

## Interpretation

The curve describes levels under intervention. The point derivative is the local slope at
dose 2.5. The elasticity rescales a local derivative and is not a separate robustness check.
The average derivative instead integrates slopes over the observed treatment law, so it
answers a population-weighted question.

Do not report the final curve point as empirically supported merely because an estimator
returned a number. Inspect `curve.support`, its diagnostics, and its warnings. Also do not
describe pointwise intervals as a simultaneous confidence band.